Import Libraries (including Random Forest)

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler,LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

# 🔹 New: Random Forest
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Load Dataset

In [3]:
file_path = "data/SLIIT_IT_Student_Data_Template_filled-updated.csv"   # your path
df = pd.read_csv(file_path)

print("First 5 rows:")
print(df.head())
print(df.columns)


First 5 rows:
   Gender District_Location Education_Stream  \
0    Male           Colombo       Bio Stream   
1  Female           Gampaha     Maths Stream   
2    Male             Kandy       Bio Stream   
3  Female             Galle     Maths Stream   
4    Male            Matara  Commerce Stream   

                                         Soft_Skills  \
0           Communication, Teamwork, Time Management   
1     Leadership, Problem Solving, Critical Thinking   
2            Public Speaking, Teamwork, Adaptability   
3  Communication, Presentation Skills, Emotional ...   
4   Time Management, Work Ethic, Attention to Detail   

                                           Key_Skils Current_semester  GPA  \
0  Java, Python, Spring Boot, MySQL, Git, RESTful...             1Y1S  1.0   
1  SQL, Python, Pandas, Power BI, Tableau, Excel ...             1Y2S  1.3   
2  HTML5, CSS3, JavaScript ES6+, React.js, Tailwi...             2Y1S  2.1   
3  Java, Spring Boot, PostgreSQL, REST APIs, Mav

Cleaning (same idea as before)

In [4]:
# 3.1 Remove unwanted 'Unnamed' columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# 3.2 Define features and target
target_col = "Experties_Preferred_Career_Field"

text_cols = ["Soft_Skills", "Key_Skils"]
cat_cols  = ["Current_semester", "Learning_Style"]
num_cols = [
    "GPA",
    "English_score",
    "Ocean_Openness",
    "Ocean_Conscientiousness",
    "Ocean_Extraversion",
    "Ocean_Agreeableness",
    "Ocean_Neuroticism",
    "Riasec_Realistic",
    "Riasec_Investigative",
    "Riasec_Artistic",
    "Riasec_Social",
    "Riasec_Enterprising",
    "Riasec_Conventional",
]


feature_cols = text_cols + cat_cols + num_cols

# Drop rows with missing target
df = df.dropna(subset=[target_col])

# Fill text missing
for c in text_cols:
    df[c] = df[c].fillna("")

# Fill categorical missing + convert to string
for c in cat_cols:
    df[c] = df[c].fillna("Unknown").astype(str)

# Convert numeric columns safely
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Drop rows where numeric values are missing
df = df.dropna(subset=num_cols)

print("Rows after cleaning:", len(df))


Rows after cleaning: 497


Features (X) and Target (y)

In [5]:
X = df[feature_cols].copy()
y = df[target_col].copy()

label_enc = LabelEncoder()
y_encoded = label_enc.fit_transform(y)

print("Original labels:", sorted(y.unique()))
print("Encoded labels:", np.unique(y_encoded))


Original labels: ['AI / Machine Learning Engineer', 'Cybersecurity / Network Engineer', 'Data Science & Analytics', 'DevOps / Cloud Engineer', 'Frontend / Web Developer', 'IT / Technology Generalist', 'Mobile App Developer', 'Software Engineer / Backend', 'UI/UX Designer']
Encoded labels: [0 1 2 3 4 5 6 7 8]


Train/Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)
print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

Train size: 397
Test size: 100


Preprocessing (same idea as logistic)

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("soft_tfidf", TfidfVectorizer(), "Soft_Skills"),
        ("key_tfidf", TfidfVectorizer(), "Key_Skils"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ],
    remainder="drop"
)


Build Random Forest Pipeline

In [8]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"   # helps if classes are imbalanced
)


Train the Random Forest

In [9]:
rf_clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("rf", rf_model)
])

print("Training Random Forest model...")
rf_clf.fit(X_train, y_train)
print("✅ Training completed")

Training Random Forest model...
✅ Training completed


Evaluate Random Forest

In [10]:
y_pred_encoded = rf_clf.predict(X_test)

rf_acc = accuracy_score(y_test, y_pred_encoded)
print("\nRandom Forest Accuracy:", rf_acc)



Random Forest Accuracy: 0.9


 Convert Predictions Back to Original Labels

In [11]:
y_test_labels = label_enc.inverse_transform(y_test)
y_pred_labels = label_enc.inverse_transform(y_pred_encoded)

print("\nClassification Report (original labels):")
print(classification_report(y_test_labels, y_pred_labels))

print("\nConfusion Matrix (encoded):")
print(confusion_matrix(y_test, y_pred_encoded))


Classification Report (original labels):
                                  precision    recall  f1-score   support

  AI / Machine Learning Engineer       1.00      1.00      1.00         7
Cybersecurity / Network Engineer       1.00      1.00      1.00         3
        Data Science & Analytics       0.77      1.00      0.87        27
         DevOps / Cloud Engineer       0.95      0.91      0.93        23
        Frontend / Web Developer       0.93      1.00      0.96        13
      IT / Technology Generalist       1.00      0.17      0.29         6
            Mobile App Developer       1.00      1.00      1.00         7
     Software Engineer / Backend       1.00      0.50      0.67         6
                  UI/UX Designer       1.00      1.00      1.00         8

                        accuracy                           0.90       100
                       macro avg       0.96      0.84      0.86       100
                    weighted avg       0.92      0.90      0.88     

Predict Career for a New Student (Random Forest)

In [12]:
new_student = pd.DataFrame([{
    "Soft_Skills": "Communication, Teamwork, Problem solving",
    "Key_Skils": "Python, SQL, Web Development",
    "Current_semester": "2Y1S",
    "Learning_Style": "Visual",
    "GPA": 3.25,
    "English_score": 78,
    "Ocean_Openness": 1,
    "Ocean_Conscientiousness": 2,
    "Ocean_Extraversion": 4,
    "Ocean_Agreeableness": 4,
    "Ocean_Neuroticism": 1,
    "Riasec_Realistic": 2,
    "Riasec_Investigative": 7,
    "Riasec_Artistic": 4,
    "Riasec_Social": 6,
    "Riasec_Enterprising": 9,
    "Riasec_Conventional": 8,
}])

pred_encoded = rf_clf.predict(new_student)[0]
pred_label = label_enc.inverse_transform([pred_encoded])[0]

print("\nPredicted Career (Random Forest):", pred_label)
print("\nRandom Forest Accuracy:", rf_acc)


Predicted Career (Random Forest): Data Science & Analytics

Random Forest Accuracy: 0.9
